### **Импорт библиотек**

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error, r2_score
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt

### **Импорт датасета**

In [ ]:
df_train = pd.read_csv('./train.csv')
df_test = pd.read_csv('./test.csv')

### **EDA**

Тренировочный сет

In [ ]:
df_train.head()

In [ ]:
df_train.tail()

In [ ]:
df_train.info()

In [ ]:
df_train.describe()

In [ ]:
df_train.shape

Тестовый датасет

In [ ]:
df_test.head()

In [ ]:
df_test.tail()

In [ ]:
df_test.info()

In [ ]:
df_test.describe()

In [ ]:
df_test.shape

### **Преобразование данных**

Обработка пропущенных значений

In [ ]:
for col in df_train.columns:
  if df_train[col].isnull().sum():
    print(f'{col}')

for col in df_test.columns:
  if df_test[col].isnull().sum():
    print(f'{col}')
# Пропущенных значений нет

Удаление дубликатов

In [ ]:
df_train.duplicated().sum()
df_test.duplicated().sum()

Удаление столбца ID

In [ ]:
df_train.drop(['ID'], inplace=True, axis=1)
df_test.drop(['ID'], inplace=True, axis=1)

Кодирование категориальных признаков

In [ ]:
X = df_train.drop(['y'], axis=1)
Y = df_train['y']
X_columns = X.columns
X

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
encoded_features = encoder.fit_transform(X[categorical_columns])
encoded_features = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))

X.drop(categorical_columns, inplace=True, axis=1)
X = pd.concat([X, encoded_features], axis=1)

In [ ]:
X

### **Тренировка сети**

Разбивка выборки

In [ ]:
import mlflow

In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")

In [ ]:
from mlflow.keras import autolog

In [ ]:
autolog(log_models=False,log_datasets=False)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

Инициализация модели

In [ ]:
model1 = keras.Sequential([
        layers.Input(shape=(X.shape[1],)),

        layers.Dense(1024, activation='sigmoid'),
        layers.Dropout(0.6),

        # layers.Dense(512, activation='relu'),
        # layers.BatchNormalization(),
        # layers.Dropout(0.5),

        # layers.Dense(256, activation='relu'),
        # layers.BatchNormalization(),
        # layers.Dropout(0.4),

        layers.Dense(128, activation='sigmoid'),
        layers.Dropout(0.3),

        layers.Dense(64, activation='sigmoid'),
        layers.Dropout(0.2),

        layers.Dense(32, activation='relu'),
        layers.Dropout(0.1),

        layers.Dense(16, activation='relu'),
        layers.Dense(1)
    ])

In [ ]:
# Модель использованая при подборе параметров и слоев. Игнорировать
# model2 = keras.Sequential([
#         layers.Input(shape=(X.shape[1],)),

#         layers.Dense(512, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.5),

#         layers.Dense(256, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.4),

#         layers.Dense(128, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.3),

#         layers.Dense(64, activation='relu'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.2),

#         layers.Dense(32, activation='relu'),
#         layers.Dense(1)
#       ])

In [ ]:
# Модель использованая при подборе параметров и слоев. Игнорировать
# model3 = keras.Sequential([
#         layers.Input(shape=(X.shape[1],)),
#         layers.Dense(256, activation='swish'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.3),

#         layers.Dense(128, activation='swish'),
#         layers.BatchNormalization(),
#         layers.Dropout(0.2),

#         layers.Dense(64, activation='swish'),
#         layers.Dense(1)
# ])

Компиляция модели

In [ ]:
model1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.999),
    loss='mse',
    metrics=['mae']
)

In [ ]:
model1.summary()

Обучение модели

In [ ]:
history = model1.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_split=0.2,
    verbose=1)

In [ ]:
train_loss, train_mae = model1.evaluate(X_train, y_train, verbose=0)
test_loss, test_mae = model1.evaluate(X_test, y_test, verbose=0)

print(f"Training MSE: {train_loss:.4f}, Training MAE: {train_mae:.4f}")
print(f"Test MSE: {test_loss:.4f}, Test MAE: {test_mae:.4f}")


Предсказание

In [ ]:
y_pred = model1.predict(X_test).flatten()

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print(f"Prediction RMSE: {rmse:.4f}")
print(f"Prediction MSE: {mse:.4f}")
print(f"Prediction R² Score: {r2:.4f}")
print(f"Prediction MAE: {mae:.4f}")

In [ ]:
with mlflow.start_run(run_id="a5cde21499c1470fb3117763ed2f10fc", experiment_id="0") as run:
    mlflow.log_metrics(
        {
            "Prediction RMSE": rmse,
            "Prediction MSE": mse,
            "Prediction R² Score": r2,
            "Prediction MAE": mae,
        }
    )

### **Resediual plot**

In [ ]:
predictions = pd.DataFrame({'True': y_test, 'Pred': y_pred})
predictions

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.residplot(x='True', y='Pred', data=predictions,ax=ax)
plt.xlabel("Истинные значения")
plt.ylabel("Отклонения предсказанных значений")
plt.tight_layout()
plt.close(fig)

In [ ]:
with mlflow.start_run(run_id="a5cde21499c1470fb3117763ed2f10fc", experiment_id="0") as run:
    mlflow.log_figure(fig, "resid_plot.png")

### **Графики потерь и метрик во время обучения**

In [ ]:
fig = plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Потеря на обучении')
plt.plot(history.history['val_loss'], label='Потеря на валидации')
plt.title('Потери модели')
plt.ylabel('Потери')
plt.xlabel('Эпоха')
plt.legend(loc='upper right')

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='MAE на обучении')
plt.plot(history.history['val_mae'], label='MAE на валидации')
plt.title('MAE модели')
plt.ylabel('MAE')
plt.xlabel('Эпоха')
plt.legend(loc='upper right')

plt.tight_layout()
plt.close(fig)

In [ ]:
with mlflow.start_run(run_id="a5cde21499c1470fb3117763ed2f10fc", experiment_id="0") as run:
    mlflow.log_figure(fig, "train-steps.png")